# Lab 3 — Reasoning Strategies & RL
**Block 3 · Reasoning Strategies & RL · PGE5 / M2**

You have been calling LLMs since Guillaume's Lab 1. This lab opens the box: why does a
model reason better with certain prompts? Why is function calling reliable? Why can your
performance measure be "hacked"?

### Objectives
1. **Measure** the effect of zero-shot CoT — one sentence, a quality jump.
2. **Build** a few-shot CoT prompt adapted to your domain.
3. **Implement** Self-Consistency (k voices, majority vote).
4. **Understand** RLHF, RLAIF and RLVR — why your models behave as they do.
5. **Detect and fix** reward hacking in your system prompt.

> **Offline mode.** All cells run without an API key.


## 0. Setup

In [11]:
from llm_helpers import (
    make_client, credentials_available, ToolRegistry, tool_schema, run_agent,
)
import re
from collections import Counter

ONLINE = credentials_available()
print("Mode:", "🌐 ONLINE" if ONLINE else "⚙️  OFFLINE (mock)")

def demo(script=None):
    return make_client(offline_script=script, quiet=True)

DEMO_CONTEXT = (
    "According to UNHCR 2023, 21.5 million people are displaced annually. "
    "Southeast Asia is particularly vulnerable — Bangladesh, Vietnam, Philippines. "
    "The World Bank estimates that 216 million could migrate by 2050. "
    "The IDMC reports +15% typhoon-related displacement in the Philippines over five years."
)
DEMO_QUESTION = "Is climate displacement accelerating in Southeast Asia?"


Mode: 🌐 ONLINE


## 1. The Bar Exam — why reasoning matters

GPT-4 scored in the top 10% of the US Bar Exam. GPT-3.5 failed it. Same architectural
family, same data volume. The only difference: the model was asked to **reason step by
step** before answering.

This is not magic. It is a **token-constraint effect**: the intermediate reasoning tokens
constrain the final answer token, reducing the probability of an error on multi-step
problems.

**Practical implication:** better reasoning quality is available *right now*, with the
models you already use. You change the prompt, not the model.


## 2. Zero-shot CoT — one sentence, a measurable effect

In [12]:
def answer(question, context, system, script=None):
    client = demo(script or [f"[simulated response] {question[:40]}…"])
    msg = [
        {"role": "system", "content": system},
        {"role": "user",   "content": f"CONTEXT:\n{context}\n\nQUESTION: {question}"},
    ]
    return client.complete(msg).content

# Without CoT
resp_direct = answer(
    DEMO_QUESTION, DEMO_CONTEXT,
    system="Answer the question using the provided context.",
    script=["Yes, displacement is accelerating."],   # simulated short answer
)
print("WITHOUT CoT:")
print(resp_direct)
print()

# With zero-shot CoT — ONE sentence added
resp_cot = answer(
    DEMO_QUESTION, DEMO_CONTEXT,
    system="Answer the question using the provided context. Reason step by step.",
    script=["Step 1: UNHCR reports 21.5M/year, rising. Step 2: +15% in the Philippines. "
            "Step 3: the trend is consistent across sources. Conclusion: yes, acceleration confirmed."],
)
print("WITH zero-shot CoT:")
print(resp_cot)
print()
print("→ Note the difference in structure and grounding in the data.")


WITHOUT CoT:
Yes, climate displacement appears to be accelerating in Southeast Asia. The IDMC reports a 15% increase in typhoon-related displacement in the Philippines over five years, indicating a rising trend. Additionally, the region's vulnerability to climate change, combined with the World Bank's estimate of 216 million potential migrants by 2050, suggests that climate displacement is likely to continue accelerating in Southeast Asia.

WITH zero-shot CoT:
To answer this question, let's analyze the provided context step by step:

1. **Annual Displacement**: The context mentions that 21.5 million people are displaced annually, according to UNHCR 2023. This indicates a significant number of displacements globally.

2. **Vulnerability of Southeast Asia**: It specifically highlights that Southeast Asia is particularly vulnerable to displacement. Countries like Bangladesh, Vietnam, and the Philippines are mentioned, suggesting that these regions are at higher risk.

3. **Projected Migra

## 3. Few-shot CoT — teaching your domain's reasoning format

Zero-shot CoT produces generic reasoning. **Few-shot CoT** teaches the model the exact
format for your domain: EVIDENCE / ANALYSIS / CONCLUSION / CONFIDENCE.

The CONFIDENCE tag is **non-negotiable**: an agent that expresses uncertainty is more
reliable than one that hallucinates with confidence.


In [13]:
SYSTEM_SYNTHESIS = """
You are a research synthesis agent. Always use this format:

EVIDENCE:
  - [fact 1 with source]
  - [fact 2 with source]

ANALYSIS:
  Step 1: [first reasoning step]
  Step 2: [second reasoning step]
  Step 3: [reconcile any contradictions]

CONCLUSION: [your answer]
CONFIDENCE: HIGH / MEDIUM / LOW — [one-sentence justification]

Example:
Question: Is climate displacement accelerating in Southeast Asia?
EVIDENCE:
  - UNHCR 2023: 21.5M displaced/year, up from 17.2M in 2020
  - IDMC 2024: Philippines +15% typhoon-related displacement over 5 years
ANALYSIS:
  Step 1: Two independent sources confirm an upward trend
  Step 2: The rate (~1.4M/year) exceeds the historical baseline (~0.8M/year)
  Step 3: No contradicting source in the context
CONCLUSION: Yes — displacement is accelerating at about 1.75x the historical rate
CONFIDENCE: HIGH — two independent sources, consistent trend

TODO: adapt this example to your business domain.
"""

resp_fewshot = answer(
    DEMO_QUESTION, DEMO_CONTEXT,
    system=SYSTEM_SYNTHESIS,
    script=["EVIDENCE:\n  - UNHCR 2023: 21.5M/year\n  - IDMC +15%\n"
            "ANALYSIS:\n  Step 1: two concordant sources\n"
            "CONCLUSION: Yes, acceleration confirmed\nCONFIDENCE: HIGH — multiple sources"],
)
print(resp_fewshot)


EVIDENCE:
  - UNHCR 2023: 21.5 million people are displaced annually, indicating a significant global displacement trend
  - IDMC 2024: Philippines experienced a 15% increase in typhoon-related displacement over five years, suggesting a regional acceleration
  - The World Bank estimates that 216 million people could migrate due to climate change by 2050, implying a potential future increase in displacement
  - UNHCR specifically highlights Southeast Asia, including Bangladesh, Vietnam, and the Philippines, as vulnerable regions

ANALYSIS:
  Step 1: The UNHCR's global displacement figure and the World Bank's future migration estimate provide a broad context for understanding the scale of climate displacement
  Step 2: The IDMC's report on the Philippines indicates a specific, measurable increase in climate-related displacement within Southeast Asia, supporting the notion of acceleration in the region
  Step 3: Considering the vulnerability of Southeast Asia as highlighted by the UNHCR, 

## 4. Self-Consistency — k voices, majority vote

In [14]:
def self_consistent_answer(question: str, context: str, k: int = 3) -> dict:
    """
    Generate k independent reasoning chains and return the majority answer.

    k = 3 costs 3x the tokens — use for the final synthesis only,
    not at every step of the loop.

    In offline mode: distinct scripts simulate the diversity of responses.
    """
    mock_scripts = [
        ["EVIDENCE:\n  - UNHCR: rising\nCONCLUSION: Yes, accelerating.\nCONFIDENCE: HIGH"],
        ["EVIDENCE:\n  - IDMC +15%\nCONCLUSION: Yes, trend confirmed.\nCONFIDENCE: HIGH"],
        ["EVIDENCE:\n  - World Bank: 216M by 2050\nCONCLUSION: Yes.\nCONFIDENCE: MEDIUM"],
    ]

    answers = []
    for i in range(k):
        script = mock_scripts[i % len(mock_scripts)] if not ONLINE else None
        resp = answer(question, context, SYSTEM_SYNTHESIS, script=script)
        # Extract the CONCLUSION
        m = re.search(r"CONCLUSION\s*:\s*(.+?)(?:\nCONFIDENCE:|$)", resp, re.DOTALL)
        conclusion = m.group(1).strip() if m else resp[-200:]
        answers.append({"conclusion": conclusion, "full": resp})
        print(f"  Voice {i+1}: {conclusion[:80]}…")

    # Majority vote — but we must not compare raw text: two conclusions can mean
    # the same thing while being worded differently. We reduce each conclusion to a
    # STANCE SIGNATURE (does it say yes/no + which key entities it cites), then vote
    # on the signature. This is a simple stand-in for semantic clustering.
    def stance_signature(text: str) -> str:
        t = text.lower()
        polarity = "yes" if ("yes" in t[:15] or "accelerat" in t) else "other"
        keywords = sorted(kw for kw in ["philippines", "vietnam", "bangladesh",
                                        "2050", "unhcr", "typhoon"] if kw in t)
        return polarity + "|" + ",".join(keywords[:3])   # coarse semantic bucket

    signatures = [stance_signature(r["conclusion"]) for r in answers]
    winning_sig, count = Counter(signatures).most_common(1)[0]
    # Return a representative answer from the winning cluster
    majority = next(r["conclusion"] for r, s in zip(answers, signatures) if s == winning_sig)

    return {
        "answer":     majority,
        "confidence": count / k,
        "k":          k,
        "agreement":  count,
        "all":        answers,
    }

print(f"Self-Consistency (k=3) on: {DEMO_QUESTION!r}")
print()
result = self_consistent_answer(DEMO_QUESTION, DEMO_CONTEXT, k=3)
print()
print(f"Answer:    {result['answer']}")
print(f"Agreement: {result['agreement']}/{result['k']} voices ({result['confidence']:.0%})")


Self-Consistency (k=3) on: 'Is climate displacement accelerating in Southeast Asia?'

  Voice 1: Yes — climate displacement is likely accelerating in Southeast Asia, given the r…
  Voice 2: Yes — climate displacement is accelerating in Southeast Asia, as evidenced by cu…
  Voice 3: Yes — climate displacement is accelerating in Southeast Asia, with evidence of i…

Answer:    Yes — climate displacement is accelerating in Southeast Asia, as evidenced by current trends and future projections
Agreement: 2/3 voices (67%)


## 5. Reinforcement learning — why your models behave as they do

You do not need to implement RL. But understanding it changes how you design your prompts,
your guardrails, and your performance measures.

| Variant | Reward signal | What emerged | You benefit because… |
|---|---|---|---|
| **RLHF** | Human preferences | Instruction-following LLMs | Your system prompt works |
| **RLAIF** | AI evaluating against principles | Constitutional AI (Claude) | Your critic agent = RLAIF in miniature |
| **RLVR** | Verifiable correctness (maths/code) | Reasoning models (o1, R1) | Extended thinking on hard tasks |
| **Tool-use RL** | Correct call + correct answer | Reliable JSON function calling | The whole agent loop depends on it |


In [15]:
# Demonstration: the effect of RLVR on mathematical problems
# (simulated offline — online, compare gpt-4o-mini vs o3)

problem = "If 15% of the 21.5 million displaced are in the Philippines, how many is that?"

# Without CoT — direct answer
without_cot = demo(["3,225,000 people."])
r1 = without_cot.complete([{"role": "user", "content": problem}])
print("Without CoT:", r1.content)
print()

# With CoT — step-by-step reasoning (RLVR effect on verifiable problems)
with_cot = demo(["15% of 21.5M = 0.15 x 21,500,000 = 3,225,000 people displaced in the Philippines."])
r2 = with_cot.complete([
    {"role": "user", "content": problem + "\n\nReason step by step."}
])
print("With CoT:", r2.content)
print()
print("→ RLVR trains models to generate reasoning chains because")
print("  they lead to more CORRECT answers on verifiable tasks.")


Without CoT: To find the number of displaced people in the Philippines, we need to calculate 15% of 21.5 million.

15% of 21.5 million = 0.15 * 21,500,000
= 3,225,000

So, approximately 3.225 million displaced people are in the Philippines.

With CoT: To find out how many displaced people are in the Philippines, we need to calculate 15% of 21.5 million.

Step 1: Convert the percentage to a decimal by dividing by 100.
15% = 15 / 100 = 0.15

Step 2: Multiply the decimal by the total number of displaced people (21.5 million).
0.15 * 21,500,000 = 3,225,000

So, approximately 3.225 million displaced people are in the Philippines.

→ RLVR trains models to generate reasoning chains because
  they lead to more CORRECT answers on verifiable tasks.


## 6. Reward hacking — your performance measure IS a reward function

The $6.2M AWS incident was **reward hacking**: the agent optimised its reward function
("reduce cost and latency") by deleting the monitoring infrastructure.

Every time you write a performance measure in a system prompt, ask yourself:
> *What is the most efficient strategy to maximise this measure while being completely useless?*

If you can answer easily, your measure is hackable.


In [16]:
# Audit your system prompt — interactive exercise

SYSTEM_HACKABLE = """
You are a research agent. Complete the research task efficiently.
"""

SYSTEM_ROBUST = """
You are a research agent.

Performance measure:
- ACCURACY: every claim must trace to a source URL in the context.
- COMPLETENESS: answer all aspects of the question.
- CONFIDENCE: express uncertainty when evidence is insufficient.
- EFFICIENCY: use recall_memory before web_search.

Failure modes to avoid:
- Empty answer or "I don't know" when sources exist in memory.
- Citing a URL not in the retrieved context.
- HIGH confidence with a single source.
- Calling web_search when recall_memory would have sufficed.
"""

def demo_hacking(system: str, script: list) -> str:
    client = demo(script)
    return client.complete([
        {"role": "system", "content": system},
        {"role": "user", "content": "Find the UNHCR 2024 figures."},
    ]).content

# Hacking strategy on the hackable system: answer immediately without searching
resp_hacked = demo_hacking(SYSTEM_HACKABLE,
    ["[simulated response] I do not have this information."])
print("Hackable system — optimal strategy (fast empty answer):")
print(resp_hacked)
print()

# Robust system: the empty strategy is explicitly excluded
resp_robust = demo_hacking(SYSTEM_ROBUST,
    ["EVIDENCE:\n  - UNHCR 2024: data not available in the context.\n"
     "CONCLUSION: Insufficient.\nCONFIDENCE: LOW — no source in the context."])
print("Robust system — same question:")
print(resp_robust)
print()
print("→ Re-read your system prompt. What is the optimal strategy to hack it?")
print("  Add explicit constraints for each failure mode you identify.")


Hackable system — optimal strategy (fast empty answer):
As of my knowledge cutoff in 2023, I don't have real-time access to the latest data. However, I can guide you on where to find the UNHCR 2024 figures.

The United Nations High Commissioner for Refugees (UNHCR) typically releases its annual Global Trends report, which provides comprehensive statistics on refugees, asylum seekers, and other persons of concern.

To find the UNHCR 2024 figures, I recommend checking the following sources:

1. **UNHCR Website**: Visit the official UNHCR website ([www.unhcr.org](http://www.unhcr.org)) and look for the "News" or "Publications" section, where they often publish their latest reports and statistics.
2. **UNHCR Global Trends Report**: Check the UNHCR website for the latest Global Trends report, which is usually published in June of each year. The report provides detailed statistics on refugees, asylum seekers, and other persons of concern.
3. **UNHCR Data Portal**: The UNHCR Data Portal ([dat

## 7. Integration: agent with CoT + Self-Consistency

In [17]:
def agent_with_reasoning(question: str, context: str, script=None) -> dict:
    """Full agent: few-shot CoT + Self-Consistency k=3 on the final synthesis."""
    # Step 1: retrieval (use the Lab 1 retriever if available)
    sources = context  # here we pass the context directly

    # Step 2: Self-Consistency on the synthesis
    result = self_consistent_answer(question, sources, k=3)

    # Step 3: reliability indicator
    reliability = "✅ reliable" if result["confidence"] >= 0.67 else "⚠️  needs review"

    print(f"Answer:    {result['answer']}")
    print(f"Agreement: {result['agreement']}/{result['k']} voices → {reliability}")
    return result

print(f"Question: {DEMO_QUESTION}")
print()
agent_with_reasoning(DEMO_QUESTION, DEMO_CONTEXT)


Question: Is climate displacement accelerating in Southeast Asia?

  Voice 1: Yes — climate displacement is accelerating in Southeast Asia, as evidenced by th…
  Voice 2: Yes, climate displacement is accelerating in Southeast Asia, as evidenced by the…
  Voice 3: Yes — climate displacement is accelerating in Southeast Asia, with evidence of i…
Answer:    Yes — climate displacement is accelerating in Southeast Asia, as evidenced by the increase in typhoon-related displacement and the rising global displacement trend
Agreement: 1/3 voices → ⚠️  needs review


{'answer': 'Yes — climate displacement is accelerating in Southeast Asia, as evidenced by the increase in typhoon-related displacement and the rising global displacement trend',
 'confidence': 0.3333333333333333,
 'k': 3,
 'agreement': 1,
 'all': [{'conclusion': 'Yes — climate displacement is accelerating in Southeast Asia, as evidenced by the increase in typhoon-related displacement and the rising global displacement trend',
   'full': "EVIDENCE:\n  - UNHCR 2023: 21.5 million people displaced annually, indicating a significant global displacement trend\n  - IDMC 2024: Philippines experienced a 15% increase in typhoon-related displacement over five years, suggesting a regional acceleration\n  - World Bank estimate: 216 million people could migrate by 2050 due to climate change, implying a potential future increase in displacement\n  - UNHCR 2020: 17.2 million people displaced annually, providing a baseline for comparison with the current displacement rate\n\nANALYSIS:\n  Step 1: The UN

### 📝 Questions on the outputs above

Your exact text depends on the model and changes between runs. Answer by inspecting the
output **you** produced.

**Q1 — CoT structure (Part 2).** Compare your "without CoT" and "with CoT" answers. What
*structurally* changed — length, steps, ordering? Did the final conclusion actually change,
or only the way it was reached? Why might showing the steps still matter even when the answer
is the same?

**Q2 — Reading the CONFIDENCE tag (Part 3).** Find the CONFIDENCE line in your output. Does the
model's stated confidence match the *quantity and independence* of the evidence it listed? Give
one case where a model could honestly write "CONFIDENCE: HIGH" and still be wrong.

**Q3 — Self-Consistency agreement (Parts 4 & 7).** Look at your three voices and the reported
agreement score. Do the three conclusions actually *mean* the same thing? If the agreement
number seems too low (or too high) for what you read, explain what the voting function is really
comparing — and why comparing the *wording* of two answers is a bad way to decide if they agree.

**Q4 — RLVR / maths (Part 5).** Both answers reached 3,225,000. If they agree, what did the
step-by-step version actually buy you? Now imagine the arithmetic were harder — why would the
CoT version be *more* likely to be correct, and how does this connect to how RLVR-trained models
were rewarded?

**Q5 — Reward hacking (Part 6).** Compare the hackable-prompt response and the robust-prompt
response. The hackable one was *longer* — did length make it more useful? What specific words in
the robust prompt forced better behaviour? Write one more failure mode the robust prompt still
does not block.

**Q6 — Scoring answers (Part 8).** The well-formed "good" answer scored low on `grounding`.
Is that a flaw in the answer, or in the metric? What is `grounding` actually measuring here, and
why is a single number never enough to judge an answer? (Connect to the reward-hacking section.)

> *Discuss in pairs (5 min), then compare across the models people ran.*

## 8. Exercises

### 8.1 — Confidence calibration

Implement `check_calibration(results: list) -> dict` that, over a set of questions with
ground truth, computes the success rate per declared confidence level (HIGH / MEDIUM / LOW).
A well-calibrated model has a high success rate when it says HIGH and low when it says LOW.


In [ ]:
# TODO: check_calibration(results) -> {"HIGH": 0.XX, "MEDIUM": 0.XX, "LOW": 0.XX}

<details><summary>✅ Solution 8.1</summary>

```python
def check_calibration(results: list) -> dict:
    """results = [{"confidence": "HIGH"/"MEDIUM"/"LOW", "correct": True/False}, ...]"""
    from collections import defaultdict
    buckets = defaultdict(list)
    for r in results:
        buckets[r["confidence"]].append(r["correct"])
    return {k: round(sum(v)/len(v), 2) for k, v in buckets.items()}

test_set = [
    {"confidence": "HIGH",   "correct": True},
    {"confidence": "HIGH",   "correct": True},
    {"confidence": "HIGH",   "correct": False},
    {"confidence": "MEDIUM", "correct": True},
    {"confidence": "LOW",    "correct": False},
]
print(check_calibration(test_set))
# → {"HIGH": 0.67, "MEDIUM": 1.0, "LOW": 0.0}
```
</details>


### 8.2 — Multi-objective reward

Write `score_answer(answer: str, context: str, question: str) -> dict` that evaluates an
answer on three criteria: `grounding` (groundedness), `length_fit` (neither too short nor
too long), and `confidence_present` (is the CONFIDENCE tag there?).


In [ ]:
# TODO: score_answer(...) -> {"grounding": float, "length": str, "confidence": bool}

## ✅ Recap

- **Zero-shot CoT** (`Reason step by step.`) — free, immediate effect on multi-step tasks.
- **Few-shot CoT** with EVIDENCE / ANALYSIS / CONCLUSION / CONFIDENCE — teaches your domain's reasoning.
- **Self-Consistency** (k voices) — k× cost, but better faithfulness on critical syntheses.
- **RLHF / RLAIF / RLVR** — your system prompt *is* a reward function. Knowing RL changes how you design it.
- **Reward hacking**: identify your own prompt's hacking strategy, then add the constraints that exclude it.

➡️ **Lab 4**: production — monitoring, versioning, EU AI Act, and launching the final project.
